In [ ]:
import numpy as np
from gfsupg.solver import CartesianGeometry, FiniteElement1D, Scipy2DFEM
from gfsupg.solver import DeC, DeCSpaceTimeSUPGSolver
# from gfsupg.solver import Numba2DFEM
from gfsupg.problem import LinearAcoustic2D
from gfsupg.plotting import *

import matplotlib.pyplot as plt

In [ ]:

#problem.T_fin = 1.
order=3

FEM1Dx = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
FEM1Dy = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
dec = DeC((order+1)//2,order,"gaussLobatto")


In [ ]:
FEM1Dx.stencil["deriv_i"]

In [ ]:
problem = LinearAcoustic2D("source_vortex", pert_coeff=0.)

In [ ]:
problem.steady_state_test

In [ ]:

Ns = np.array([20,20], dtype=np.int32)

geom = CartesianGeometry(problem.xL,problem.xR, Ns, problem.geometry_folder, BC=problem.BC)

In [ ]:
FEM2D = Scipy2DFEM(geom,FEM1Dx, FEM1Dy, folder=problem.folderName)

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec)

In [ ]:
solver.set_ic()

In [ ]:
plot_one_sol(problem, FEM2D, solver.ic_vect, levels=100)

zzz = FEM2D.operator["IDx_tilde"]@solver.ic_vect["u"]+FEM2D.operator["IDy_tilde"]@solver.ic_vect["v"]-\
  FEM2D.operator["mass_tilde"]@FEM2D.evaluate_function(problem.p_source_fun)

np.min(zzz)

## Davide's approach

In [ ]:
from scipy.optimize import LinearConstraint, minimize
from scipy.sparse import hstack, vstack, identity
div_oper = hstack([FEM2D.operator["IDx_tilde"],FEM2D.operator["IDy_tilde"]] )
# div_oper = hstack([FEM2D.operator["IDx"],FEM2D.operator["IDy"]] )

linear_constraint = LinearConstraint(div_oper , np.zeros(div_oper.shape[0]), np.zeros(div_oper.shape[0]))


u = solver.ic_vect["u"]
v = solver.ic_vect["v"]
IC_vect = solver.ic_vect

u_ic = np.concatenate([u,v])
u_ic.shape

def target_function(u):
    return np.sum((u-u_ic)**2)
def target_function_dir(u):
    return 2.*(u-u_ic)
def target_function_hess(u):
    return 2.*identity(len(u))


res = minimize(target_function, u_ic, method='trust-constr',\
                jac=target_function_dir, hess=target_function_hess,
               constraints=[linear_constraint],
               options={'verbose': 1})


q_vec = dict()
q_vec["u"]       = res.x[:len(u)]
q_vec["v"]       = res.x[len(u):]



# Pressure for coriolis
cor = problem.coriolis
linear_constraint_v = \
    LinearConstraint(FEM2D.operator["IDx"], \
                        cor*q_vec["v"], \
                        cor*q_vec["v"])
linear_constraint_u = \
    LinearConstraint(FEM2D.operator["IDy"], \
                        -cor*q_vec["u"], \
                        -cor*q_vec["u"])

def target_function(p):
    return np.sum((p-IC_vect["p"])**2)
def target_function_dir(p):
    return 2.*(p-IC_vect["p"])
def target_function_hess(p):
    return 2.*identity(len(p))


res = minimize(target_function, IC_vect["p"], method='trust-constr',\
                jac=target_function_dir, hess=target_function_hess,
            constraints=[linear_constraint_u, linear_constraint_v],
            options={'verbose': 1})

q_vec["p"]       = res.x

In [ ]:
q_davide_vec = dict()
q_davide_vec["u"]       = res.x[:len(u)]
q_davide_vec["v"]       = res.x[len(u):]

from src.processing import divfree_projection_optimization, divfree_projection_integration

q_davide_vec = divfree_projection_optimization(FEM2D, solver.ic_vect)

div_davide_vec = FEM2D.compute_discrete_divergence(q_davide_vec)

div_davide_mat = FEM2D.vect_to_mat(div_davide_vec)

print(np.mean(np.abs(div_davide_mat[:-1,:-1])))
print(np.mean(np.abs(div_davide_mat)))

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], div_davide_mat)
plt.colorbar()
plt.show()

div_NGF = (FEM2D.operator["IDx"]@q_davide_vec["u"]+FEM2D.operator["IDy_tilde"]@q_davide_vec["v"])/geom.dx[0]/geom.dx[1]

div_NGF_mat = FEM2D.vect_to_mat(div_NGF)

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], div_NGF_mat)
plt.colorbar()
plt.show()


In [ ]:
u_mat = FEM2D.vect_to_mat(u)
v_mat = FEM2D.vect_to_mat(v)

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mat)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], FEM2D.vect_to_mat(q_davide_vec["u"]))
plt.colorbar()
plt.show()


plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], v_mat)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], FEM2D.vect_to_mat(q_davide_vec["v"]))
plt.colorbar()
plt.show()
plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], v_mat-FEM2D.vect_to_mat(q_davide_vec["v"]))
plt.colorbar()
plt.show()


# Unconstrained Optimization

In [ ]:
from scipy.optimize import LinearConstraint, minimize
from scipy.sparse import hstack, vstack, identity
# I want to put more constraints here in a weak sense

div_oper = hstack([FEM2D.operator["IDx_tilde"],FEM2D.operator["IDy_tilde"]] )
# div_oper = hstack([FEM2D.operator["IDx"],FEM2D.operator["IDy"]] )

dx_div_oper = hstack([FEM2D.operator["DxDx_tilde"],FEM2D.operator["DxDy_tilde"]])
dy_div_oper = hstack([FEM2D.operator["DyDx_tilde"],FEM2D.operator["DyDy_tilde"]])
grad_div_oper = vstack([dx_div_oper, dy_div_oper])

linear_constraint = LinearConstraint(div_oper , np.zeros(div_oper.shape[0]), np.zeros(div_oper.shape[0]))


mass_oper = vstack([
    hstack([   FEM2D.operator["mass"], 0.*FEM2D.operator["mass"]]),\
    hstack([0.*FEM2D.operator["mass"],    FEM2D.operator["mass"]])
])
        
mass_mass = mass_oper.T @ mass_oper
grad_div_grad_div = grad_div_oper.T@grad_div_oper

u = solver.ic_vect["u"]
v = solver.ic_vect["v"]
IC_vect = solver.ic_vect

u_ic = np.concatenate([u,v])
N = u_ic.shape[0]

pen_grad_div = 1e5

def target_function(u):
    return np.sum((mass_oper@(u-u_ic))**2) + pen_grad_div*np.sum((grad_div_oper@u)**2)
def target_function_dir(u):
    return 2.*mass_mass@(u-u_ic)+ pen_grad_div*2.* grad_div_grad_div@u
def target_function_hess(u):
    return 2.*mass_mass+pen_grad_div*2. * grad_div_grad_div


# res = minimize( target_function, u_ic,  method='trust-ncg',
#                jac=target_function_dir, hess=target_function_hess,
#                options={'disp': True})

res = minimize(target_function, u_ic, method='trust-constr',\
                jac=target_function_dir, hess=target_function_hess,
               constraints=[linear_constraint],
               options={'verbose': 1})


q_vec = dict()
q_vec["u"]       = res.x[:len(u)]
q_vec["v"]       = res.x[len(u):]



In [ ]:
target_function(res.x)

In [ ]:
np.max(np.abs(grad_div_oper@res.x))

## Mario's approach

In [ ]:
u = solver.ic_vect["u"]
v = solver.ic_vect["v"]

u_mat = FEM2D.vect_to_mat(u)
v_mat = FEM2D.vect_to_mat(v)

u_mario = np.empty_like(u_mat)
v_mario = np.empty_like(v_mat)


# Solve I^y u(x_m) l = int y_0^y_l u^ex(x_m,y) dy

from src.quadr import nodes_weights
HO_quad_nodes, HO_quad_weights = nodes_weights(10, "gaussLegendre")

RHS = np.zeros(FEM1Dy.degree)
for m, x in enumerate(FEM2D.xx_dofs[0]):
   u_mario[m, 0] = problem.ics["u"](x, geom.xL[1])

   # Solve I^y u(x_m) l = int y_0^y_l u^ex(x_m,y) dy
   for j_cell in range(geom.N_elem_dir[1]):
      j_cell_global = j_cell*FEM1Dy.degree      
      j1_cell_global = (j_cell+1)*FEM1Dy.degree
      for j_dof in range(1,FEM1Dy.degree+1):
         j_dof_global = j_cell_global+j_dof 
         dy = FEM2D.xx_dofs[1][j_dof_global] - geom.xx[1][j_cell]
         quad_nodes = HO_quad_nodes*dy +geom.xx[1][j_cell]
         quad_weights =  HO_quad_weights*dy
         RHS[j_dof-1] = np.sum(np.vectorize(problem.ics["u"])(x, quad_nodes) * quad_weights)
      
      dy = geom.xx[1][j_cell+1]-geom.xx[1][j_cell]
      RHS = RHS/dy- FEM1Dy.matrix["int_mat"][1:,0]*u_mario[m, j_cell_global]
      u_mario[ m, j_cell_global+1:j1_cell_global+1 ] = np.linalg.solve(FEM1Dy.matrix["int_mat"][1:,1:],RHS)




# Solve I^x v(y_l) m = int x_0^x_m v^ex(x,y_l) dx

RHS = np.zeros(FEM1Dx.degree)
for l, y in enumerate(FEM2D.xx_dofs[1]):
   v_mario[0,l] = problem.ics["v"](geom.xL[0], y )

   # Solve I^x v(y_l) m = int x_0^x_m v^ex(x,y_l) dx
   for j_cell in range(geom.N_elem_dir[0]):
      j_cell_global = j_cell*FEM1Dx.degree      
      j1_cell_global = (j_cell+1)*FEM1Dx.degree
      for j_dof in range(1,FEM1Dx.degree+1):
         j_dof_global = j_cell_global+j_dof 
         dx = FEM2D.xx_dofs[0][j_dof_global] - geom.xx[0][j_cell]
         quad_nodes = HO_quad_nodes*dx +geom.xx[0][j_cell]
         quad_weights =  HO_quad_weights*dx
         RHS[j_dof-1] = np.sum(np.vectorize(problem.ics["v"])(quad_nodes, y) * quad_weights)
      
      dx = geom.xx[0][j_cell+1]-geom.xx[0][j_cell]
      RHS = RHS/dx- FEM1Dx.matrix["int_mat"][1:,0]*v_mario[ j_cell_global, l]
      v_mario[ j_cell_global+1:j1_cell_global+1, l ] = np.linalg.solve(\
         FEM1Dx.matrix["int_mat"][1:,1:],RHS)




In [ ]:
q_mario_vec = dict()
q_mario_vec["u"]       = FEM2D.mat_to_vect(u_mario)
q_mario_vec["v"]       = FEM2D.mat_to_vect(v_mario)

q_mario_vec = divfree_projection_integration(FEM2D, solver.ic_vect, problem.ics) 

div_mario_vec = FEM2D.compute_discrete_divergence(q_mario_vec)

div_mario_mat = FEM2D.vect_to_mat(div_mario_vec)

print(np.mean(np.abs(div_mario_mat[:-1,:-1])))
print(np.mean(np.abs(div_mario_mat)))

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], div_mario_mat)
plt.colorbar()
plt.show()

div_NGF = (FEM2D.operator["IDx"]@q_mario_vec["u"]+FEM2D.operator["IDy_tilde"]@q_mario_vec["v"])/geom.dx[0]/geom.dx[1]

div_NGF_mat = FEM2D.vect_to_mat(div_NGF)

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], div_NGF_mat)
plt.colorbar()
plt.show()


In [ ]:


plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mat)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mario)
plt.colorbar()
plt.show()


plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], v_mat)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], v_mario)
plt.colorbar()
plt.show()
plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], v_mat-v_mario)
plt.colorbar()
plt.show()


## Wasilij's approach

In [ ]:
u = solver.ic_vect["u"]
v = solver.ic_vect["v"]

u_mat = FEM2D.vect_to_mat(u)
v_mat = FEM2D.vect_to_mat(v)

U_mat = np.zeros(u_mat.shape)
V_mat = np.zeros(u_mat.shape)

for i in range(geom.N_elem_dir[0]):
   i_idx_left  = i*FEM2D.FEM1Dx.degree
   i_idx_right = (i+1)*FEM2D.FEM1Dx.degree


   for j in range(geom.N_elem_dir[1]):
      j_idx_bottom  = j*FEM2D.FEM1Dy.degree
      j_idx_top     = (j+1)*FEM2D.FEM1Dy.degree

      
      EE = FEM1Dx.matrix["eval_mat"]
      II = FEM1Dy.matrix["int_mat"]*geom.dx[1]
      uu = u_mat[i_idx_left:i_idx_right+1,\
         j_idx_bottom:j_idx_top+1]



      U_mat[ i_idx_left:i_idx_right+1,\
         j_idx_bottom:j_idx_top+1 ] = \
      U_mat[i_idx_left:i_idx_right+1, j_idx_bottom].reshape((-1,1)) \
         + np.einsum('zt,wr,tr->zw',EE,II,uu)


for j in range(geom.N_elem_dir[1]):
   j_idx_bottom  = j*FEM2D.FEM1Dy.degree
   j_idx_top     = (j+1)*FEM2D.FEM1Dy.degree
   for i in range(geom.N_elem_dir[0]):
      i_idx_left  = i*FEM2D.FEM1Dx.degree
      i_idx_right = (i+1)*FEM2D.FEM1Dx.degree
      
      II = FEM1Dx.matrix["int_mat"]*geom.dx[0]
      EE = FEM1Dy.matrix["eval_mat"]
      vv = v_mat[i_idx_left:i_idx_right+1,\
         j_idx_bottom:j_idx_top+1]
      V_mat[ i_idx_left:i_idx_right+1,\
         j_idx_bottom:j_idx_top+1 ] = \
      V_mat[i_idx_left, j_idx_bottom:j_idx_top+1].reshape((1,-1))  \
         + np.einsum('zt,wr,tr->zw',II,EE,vv)
      # if j==0:
      #    print(np.einsum('zt,wr,tr->zw',II,EE,vv))
         # print(V_mat[ i_idx_left:i_idx_right+1,\
         # j_idx_bottom:j_idx_top+1 ])
         # print("")



In [ ]:

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mat+v_mat.T)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_mat+V_mat.T)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_mat)
plt.colorbar()
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], V_mat)
plt.colorbar()
plt.show()

In [ ]:
levels=np.linspace(np.min(U_mat),np.max(U_mat),100)

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_mat, levels=levels)
plt.colorbar()
plt.title("U")
plt.show()

levels=np.linspace(np.min(V_mat),np.max(V_mat),100)
plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], V_mat, levels=levels)
plt.colorbar()
plt.title("V")
plt.show()
plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_mat+V_mat)
plt.colorbar()
plt.title("U+V")
plt.show()


In [ ]:
U_vec       = FEM2D.mat_to_vect(U_mat)
V_vec       = FEM2D.mat_to_vect(V_mat)

levels=np.linspace(np.min(u),np.max(u),100)


u_tilde_vec = FEM2D.operator["inv_lump"] @ FEM2D.operator["IDy"] @ U_vec
v_tilde_vec = FEM2D.operator["inv_lump"] @ FEM2D.operator["IDx"] @ V_vec

u_tilde = FEM2D.vect_to_mat(u_tilde_vec)
v_tilde = FEM2D.vect_to_mat(v_tilde_vec)


u_diff = FEM2D.operator["mass"]@u - FEM2D.operator["IDy"] @ U_vec

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_tilde, levels=levels)
plt.colorbar()
plt.title("M^{-1}DIu")
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mat,levels=levels)
plt.colorbar()
plt.title("u")
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mat-u_tilde)
plt.colorbar()
plt.title("u-M^{-1}DIu")
plt.show()


plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], FEM2D.vect_to_mat(u_diff))
plt.colorbar()
plt.title("Mu-DIu")
plt.show()

In [ ]:
delta = np.zeros(np.shape(U_mat))
delta[:,:] = U_mat+V_mat -U_mat[:,0]-V_mat[0,:]

U_tilde = U_mat-delta/2
V_tilde = V_mat-delta/2

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_mat)
plt.colorbar()
plt.title("U")
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_mat+V_mat)
plt.colorbar()
plt.title("U +V")
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], U_tilde+V_tilde)
plt.colorbar()
plt.title(r"$\tilde{U} +\tilde{V}$")
plt.show()

U_vec       = FEM2D.mat_to_vect(U_mat)
V_vec       = FEM2D.mat_to_vect(V_mat)

U_tilde_vec = FEM2D.mat_to_vect(U_tilde)
V_tilde_vec = FEM2D.mat_to_vect(V_tilde)

u_tilde_vec = FEM2D.operator["inv_lump"] @ FEM2D.operator["IDy"]@U_vec
v_tilde_vec = FEM2D.operator["inv_lump"] @ FEM2D.operator["IDx"]@V_vec

u_tilde = FEM2D.vect_to_mat(u_tilde_vec)
v_tilde = FEM2D.vect_to_mat(v_tilde_vec)


plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_tilde)
plt.colorbar()
plt.title(r"$\tilde{u}$")
plt.show()

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_mat)
plt.colorbar()
plt.title(r"$u$")
plt.show()



plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], u_tilde-u_mat)
plt.colorbar()
plt.title(r"Difference $u-\tilde{u}$")
plt.show()

q=dict()
q["u"]  = u_tilde_vec
q["v"]  = v_tilde_vec
div_tilde = FEM2D.compute_discrete_divergence(q)

plt.contourf(FEM2D.xx_mat[0], FEM2D.xx_mat[1], FEM2D.vect_to_mat(div_tilde))
plt.colorbar()
plt.title(r"$\widetilde{\nabla}\cdot \tilde{u}$")
plt.show()

In [ ]:
FEM1Dy.matrix["int_mat"]

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec)
#solver.set_save_slabs(200)
solver.set_Nt_max(1e6)
q_save, tt_save, comp_time, err, err_vertex  = solver.solve(with_error = True, with_error_vertex = True, GF=False, stab_coeff=0.05)

div_errors = FEM2D.compute_discrete_divergence_norms(q_save)
np.mean(np.abs(div_errors))


In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec)
q_save_GF, tt_save_GF, comp_time_GF, err_GF, err_GF_vertex = solver.solve(with_error = True, with_error_vertex = True, GF=True, SUPG_coeff=0.05)
print(err_GF)
print(err_GF_vertex)

div_errors_GF = FEM2D.compute_discrete_divergence_norms(q_save_GF)
np.mean(np.abs(div_errors))

In [ ]:
plt.semilogy(tt_save, div_errors,label="SUPG")
plt.semilogy(tt_save_GF, div_errors_GF,label="SUPG-GF")
plt.legend()


In [ ]:
fig,axs = plt.subplots(1,3,figsize=(15,4))
plot_sol(FEM2D, solver.ic_vect["u"], axs[0], fig)
plot_sol(FEM2D, solver.ic_vect["v"], axs[1], fig)
plot_sol(FEM2D, solver.ic_vect["p"], axs[2], fig, levels=np.linspace(np.min(solver.ic_vect["p"]),np.max(solver.ic_vect["p"]),13))
for iv, var in enumerate(problem.vars):
    axs[iv].set_title(var)

In [ ]:

plot_all_sols(problem, FEM2D, q_save, len(q_save)-1, tt_save[len(q_save)-1], levels=None)
plot_all_sols(problem, FEM2D, q_save, len(q_save_GF)-1, tt_save_GF[len(q_save_GF)-1], levels=None)


In [ ]:
diff   = dict()
diffGF = dict()
for var in problem.vars:
    diff[var] = q_save[var]-solver.ic_no_pert[var]
    diffGF[var]   = q_save_GF[var]-solver.ic_no_pert[var]
v2   = q_save["u"][-1,:]**2+q_save["v"][-1,:]**2
v2GF = q_save_GF["u"][-1,:]**2+q_save_GF["v"][-1,:]**2
v2IC = solver.ic_no_pert["u"]**2+ solver.ic_no_pert["v"]**2

diffv2   = v2  -v2IC
diffv2GF = v2GF-v2IC

plot_all_sols(problem, FEM2D, diff, len(q_save)-1, tt_save[len(q_save)-1], levels=None)
plot_all_sols(problem, FEM2D, diffGF, len(q_save_GF)-1, tt_save_GF[len(q_save_GF)-1], levels=None)

fig, axs = plt.subplots(1,3, figsize=(15,4))
plot_sol(FEM2D, np.abs(diffv2) , axs[0],fig, levels=21)
plot_sol(FEM2D, np.abs(diffv2GF),axs[1],fig, levels=21)
plot_sol(FEM2D, np.abs(v2IC),axs[2],fig, levels=21)
axs[0].set_title("Non Global Flux")
axs[1].set_title("Global Flux")
plt.show()

it = -1
fig, axs = plt.subplots(1,3, figsize=(15,4))
plot_sol(FEM2D, diff["u"][it,:]**2+diff["v"][it,:]**2 , axs[0],fig, levels=21)
plot_sol(FEM2D, diffGF["u"][it,:]**2+diffGF["v"][it,:]**2,axs[1],fig, levels=21)
plot_sol(FEM2D, np.abs(v2IC),axs[2],fig, levels=21)
axs[0].set_title("Non Global Flux")
axs[1].set_title("Global Flux")
plt.show()

In [ ]:
from numba.experimental import jitclass
from numba import types, typed
import numba



@numba.njit
def get_stencil_indexes(i_cell, degree):
    jl = (i_cell-1)*degree +degree
    jr = (i_cell+2)*degree +degree
    return jl, jr

stenc_specs = [
    ('stencil_x',types.float64[:,:]),
    ('stencil_y',types.float64[:,:])
]
@jitclass(stenc_specs)
class Stencil_x_y:
    def __init__(self,stencil_x,stencil_y):
        self.stencil_x = stencil_x
        self.stencil_y = stencil_y

@numba.njit
def invert_nnz(ll):
    zz=np.zeros(ll.shape)
    zz = (ll!=0)/(ll+1e-100)
    return zz


xx_dofs_tv = (types.int32, types.float64[:])
ndof_tv = (types.int32, types.int32)

FEM2D_specs=[
    ('FEM1Dx',FiniteElement1D.class_type.instance_type),
    ('FEM1Dy',FiniteElement1D.class_type.instance_type),
    ('geom',  CartesianGeometry.class_type.instance_type),
    ('xx_dofs',  types.DictType(*xx_dofs_tv)),
    ('n_dof_dir',types.DictType(*ndof_tv)),
    ('vect_2_mat',numba.types.int32[:,:]),
    ('mat_2_vect',numba.types.int32[:,:]),
    ('n_dof_tot',numba.types.int32),
    ('mass', Stencil_x_y.class_type.instance_type),
    ('lump_mass', Stencil_x_y.class_type.instance_type),
    ('IDx', Stencil_x_y.class_type.instance_type),
    ('IDy', Stencil_x_y.class_type.instance_type),
    ('DxI', Stencil_x_y.class_type.instance_type),
    ('DyI', Stencil_x_y.class_type.instance_type),
    ('IDxy', Stencil_x_y.class_type.instance_type),
    ('DxDx', Stencil_x_y.class_type.instance_type),
    ('DyDy', Stencil_x_y.class_type.instance_type),
    ('DxDy', Stencil_x_y.class_type.instance_type),
    ('DyDx', Stencil_x_y.class_type.instance_type),
    ('DxDxy', Stencil_x_y.class_type.instance_type),
    ('DyDxy', Stencil_x_y.class_type.instance_type),
    ('DxDx_tilde', Stencil_x_y.class_type.instance_type),
    ('DyDy_tilde', Stencil_x_y.class_type.instance_type),
    ('DxDy_tilde', Stencil_x_y.class_type.instance_type),
    ('DyDx_tilde', Stencil_x_y.class_type.instance_type),
    ('IDx_tilde', Stencil_x_y.class_type.instance_type),
    ('IDy_tilde', Stencil_x_y.class_type.instance_type),
    ('inv_lump', Stencil_x_y.class_type.instance_type)
]
@jitclass(FEM2D_specs)
class Numba2DFEM:
    def __init__(self, geom, FEM1Dx, FEM1Dy=None):
        self.FEM1Dx = FEM1Dx
        if FEM1Dy is None:
            self.FEM1Dy = FEM1Dx
        else:
            self.FEM1Dy = FEM1Dy

        self.geom = geom
        self.xx_dofs = typed.Dict.empty(*xx_dofs_tv)
        self.n_dof_dir = typed.Dict.empty(*ndof_tv)
        for k in range(geom.dim):
            if k==0:
                FEM1D = self.FEM1Dx
            elif k==1:
                FEM1D = self.FEM1Dy
            xx_dofs_mat = np.zeros((len(self.geom.xx[k]), len(self.geom.dx[k]*FEM1D.nodes[:-1])))
            for i in range(len(self.geom.xx[k])):
                xx_dofs_mat[i,:] = self.geom.xx[k][i] + self.geom.dx[k]*FEM1D.nodes[:-1]
            self.xx_dofs[k]   = np.reshape(xx_dofs_mat,-1)

            if self.geom.BC[0]!=0: # non periodic case
                if FEM1D.degree>1:
                    for z in range(FEM1D.degree-1):
                        self.xx_dofs[k] = np.delete(self.xx_dofs[k], -1)
            self.n_dof_dir[k] = len(self.xx_dofs[k] )
        self.n_dof_tot = self.n_dof_dir[0]*self.n_dof_dir[1]
        self.vect_matr_maps()
        self.assemble2D_stencils()

    def vect_matr_maps(self):
        self.vect_2_mat = np.zeros((self.n_dof_tot, 2),dtype=np.int32)
        self.mat_2_vect = np.zeros((self.n_dof_dir[0],self.n_dof_dir[1]), dtype=np.int32)
        global_idx = -1
        for ix in range(self.n_dof_dir[0]):
            xi = self.xx_dofs[0][ix]
            for iy in range(self.n_dof_dir[1]):
                yi = self.xx_dofs[1][iy]
                global_idx +=1
                self.vect_2_mat[global_idx,:] = [ix,iy]
                self.mat_2_vect[ix,iy]        = global_idx
                
    # def vect_to_mat(self, u_vec):
    #     u_mat = np.zeros((self.n_dof_dir[0],self.n_dof_dir[1]))
    #     for i in range(self.n_dof_tot):
    #         u_mat[self.vect_2_mat[i,0],self.vect_2_mat[i,1]] =\
    #             u_vec[i]
    #     return u_mat

    # def mat_to_vect(self, u_mat):
    #     u_vec = np.zeros(self.n_dof_tot)
    #     for i in range(self.n_dof_tot):
    #         u_vec[i] = u_mat[self.vect_2_mat[i,0],self.vect_2_mat[i,1]]
    #     return u_vec
    
    def extend_vec(self,u):
        degree_x = self.FEM1Dx.degree
        degree_y = self.FEM1Dy.degree
        Nx = self.n_dof_dir[0]
        Ny = self.n_dof_dir[1]
        if self.geom.BC[0] == 0:  # periodic in x
            Nx_extend = Nx + 2*degree_x  #one extra cell on the right and one on the left
        else:
            Nx_extend = Nx + 3*degree_x -1 # two extra cells on the right (one dof already contained)
        if self.geom.BC[1] == 0:  # periodic in x
            Ny_extend = Ny + 2*degree_y  #one extra cell on the right and one on the left
        else:
            Ny_extend = Ny + 3*degree_y -1 # two extra cells on the right (one dof already contained)
        u_ext = np.zeros((Nx_extend, Ny_extend))
        u_ext[degree_x:Nx+degree_x,degree_y:Ny+degree_y] = u
        if self.geom.BC[0] == 0:  # periodic in x
            for i in range(degree_x):
                u_ext[i,degree_y:-degree_y]    = u[-degree_x+i,:]
                u_ext[-i-1,degree_y:-degree_y] = u[degree_x-i-1,:]
        else:  #non periodic in x
            for i in range(degree_x):
                u_ext[i,degree_y:Ny+degree_y] = u[0,:]
            for i in range(2*degree_x-1):
                u_ext[-i-1,degree_y:Ny+degree_y] = u[-1,:]
        if self.geom.BC[1] == 0: #periodic in y
            for i in range(degree_y):
                u_ext[:,i] = u_ext[:,-2*degree_y+i]
                u_ext[:,-i-1] = u_ext[:,2*degree_y-1-i]
        else:
            for i in range(degree_y):
                u_ext[:,i] = u_ext[:,degree_y]
            for i in range(2*degree_y-1):
                u_ext[:,-i-1] = u_ext[:,Ny+degree_y-1]
        return u_ext


    def stencils_product(self, stencil_x_y, u):
        stencil_x = stencil_x_y.stencil_x
        stencil_y = stencil_x_y.stencil_y
        degree_x = self.FEM1Dx.degree
        degree_y = self.FEM1Dy.degree
        pr = np.zeros(u.shape)
        u_ext = self.extend_vec(u) 
        for ix_cell in range(self.geom.N_elem_dir[0]):
            for ix_dof in range(degree_x):
                for iy_cell in range(self.geom.N_elem_dir[1]):
                    for iy_dof in range(degree_y):
                        jxl, jxr = get_stencil_indexes(ix_cell, degree_x)
                        jyl, jyr = get_stencil_indexes(iy_cell, degree_y)                    
                        tmp_pr = stencil_x[ix_dof,:].T@\
                                u_ext[jxl:jxr, jyl:jyr]@\
                                stencil_y[iy_dof,:]
                        pr[ix_cell*degree_x+ix_dof,iy_cell*degree_y+iy_dof] = tmp_pr
                if self.geom.BC[1]!=0: # non periodic in y
                    iy_cell = self.geom.N_elem_dir[1]
                    iy_dof  = 0
                    jxl, jxr = get_stencil_indexes(ix_cell, degree_x)
                    jyl, jyr = get_stencil_indexes(iy_cell, degree_y)                   
                    tmp_pr = stencil_x[ix_dof,:].T@\
                            u_ext[jxl:jxr, jyl:jyr]@\
                            stencil_y[iy_dof,:]
                    pr[ix_cell*degree_x+ix_dof,iy_cell*degree_y+iy_dof] = tmp_pr
        if self.geom.BC[0]!=0: # non periodic in x
            ix_cell = self.geom.N_elem_dir[0]
            ix_dof = 0
            for iy_cell in range(self.geom.N_elem_dir[1]):
                for iy_dof in range(degree_y):
                    jxl, jxr = get_stencil_indexes(ix_cell, degree_x)
                    jyl, jyr = get_stencil_indexes(iy_cell, degree_y)                    
                    tmp_pr = stencil_x[ix_dof,:].T@\
                            u_ext[jxl:jxr, jyl:jyr]@\
                            stencil_y[iy_dof,:]
                    pr[ix_cell*degree_x+ix_dof,iy_cell*degree_y+iy_dof] = tmp_pr
            if self.geom.BC[1]!=0: # non periodic in y
                iy_cell = self.geom.N_elem_dir[1]
                iy_dof  = 0
                jxl, jxr = get_stencil_indexes(ix_cell, degree_x)
                jyl, jyr = get_stencil_indexes(iy_cell, degree_y)                
                tmp_pr = stencil_x[ix_dof,:].T@\
                        u_ext[jxl:jxr, jyl:jyr]@\
                        stencil_y[iy_dof,:]
                pr[ix_cell*degree_x+ix_dof,iy_cell*degree_y+iy_dof] = tmp_pr
        return pr    

    def assemble2D_stencils(self):
        self.mass       = Stencil_x_y(self.FEM1Dx.stencil["mass"]     *self.geom.dx[0],self.FEM1Dy.stencil["mass"]     *self.geom.dx[1])     
        self.lump_mass  = Stencil_x_y(self.FEM1Dx.stencil["lump_mass"]*self.geom.dx[0],self.FEM1Dy.stencil["lump_mass"]*self.geom.dx[1])
        self.IDx        = Stencil_x_y(self.FEM1Dx.stencil["deriv_j"]                  ,self.FEM1Dy.stencil["mass"]     *self.geom.dx[1])     
        self.IDy        = Stencil_x_y(self.FEM1Dx.stencil["mass"]     *self.geom.dx[0],self.FEM1Dy.stencil["deriv_j"]                  )  
        self.DxI        = Stencil_x_y(self.FEM1Dx.stencil["deriv_i"]                  ,self.FEM1Dy.stencil["mass"]     *self.geom.dx[1])     
        self.DyI        = Stencil_x_y(self.FEM1Dx.stencil["mass"]     *self.geom.dx[0],self.FEM1Dy.stencil["deriv_i"]                  )  
        self.IDxy       = Stencil_x_y(self.FEM1Dx.stencil["deriv_j"]                  ,self.FEM1Dy.stencil["deriv_j"]                  )  
        self.DxDx       = Stencil_x_y(self.FEM1Dx.stencil["deriv_ij"] /self.geom.dx[0],self.FEM1Dy.stencil["mass"]     *self.geom.dx[1])     
        self.DyDy       = Stencil_x_y(self.FEM1Dx.stencil["mass"]     *self.geom.dx[0],self.FEM1Dy.stencil["deriv_ij"] /self.geom.dx[1]) 
        self.DxDy       = Stencil_x_y(self.FEM1Dx.stencil["deriv_i"]                  ,self.FEM1Dy.stencil["deriv_j"]                  )  
        self.DyDx       = Stencil_x_y(self.FEM1Dx.stencil["deriv_j"]                  ,self.FEM1Dy.stencil["deriv_i"]                  )  
        self.DxDxy      = Stencil_x_y(self.FEM1Dx.stencil["deriv_ij"] /self.geom.dx[0],self.FEM1Dy.stencil["deriv_j"]                  )  
        self.DyDxy      = Stencil_x_y(self.FEM1Dx.stencil["deriv_j"]                  ,self.FEM1Dy.stencil["deriv_ij"] /self.geom.dx[1])                      
        self.DxDx_tilde = Stencil_x_y(self.FEM1Dx.stencil["deriv_ij_tilde"]/self.geom.dx[0],self.FEM1Dy.stencil["der_int_tilde"]*self.geom.dx[1]) 
        self.DyDy_tilde = Stencil_x_y(self.FEM1Dx.stencil["der_int_tilde"] *self.geom.dx[0],self.FEM1Dy.stencil["deriv_ij_tilde"]/self.geom.dx[1])
        self.DxDy_tilde = Stencil_x_y(self.FEM1Dx.stencil["deriv_i_tilde"]                 ,self.FEM1Dy.stencil["deriv_j_bar"])
        self.DyDx_tilde = Stencil_x_y(self.FEM1Dx.stencil["deriv_j_bar"]                   ,self.FEM1Dy.stencil["deriv_i_tilde"])
        self.IDx_tilde  = Stencil_x_y(self.FEM1Dx.stencil["deriv_j_bar"]                   ,self.FEM1Dy.stencil["der_int_tilde"]*self.geom.dx[1])                 
        self.IDy_tilde  = Stencil_x_y(self.FEM1Dx.stencil["der_int_tilde"]*self.geom.dx[0] ,self.FEM1Dy.stencil["deriv_j_bar"])   
        self.inv_lump   = Stencil_x_y(invert_nnz(self.lump_mass.stencil_x), invert_nnz(self.lump_mass.stencil_y))



In [ ]:
FEM2Dn = Numba2DFEM(geom, FEM1Dx, FEM1Dy)
q0 = dict()
for var in problem.vars:
    q0[var] = np.array([
        [
            problem.ics[var](x,y) for y in FEM2Dn.xx_dofs[1]
        ] for x in FEM2Dn.xx_dofs[0]
    ])
print(q0["u"].shape)
uu_ext = FEM2Dn.extend_vec(q0["u"])
print(uu_ext.shape)
# uu_mat = FEM2Dn.vect_to_mat(q0["u"])
# uu_vec = FEM2Dn.mat_to_vect(uu_mat)
print(FEM1Dx.stencil["mass"].shape)
mass = Stencil_x_y(FEM1Dx.stencil["mass"],FEM1Dy.stencil["mass"])
%time zzz = FEM2Dn.stencils_product(FEM2Dn.mass,q0["u"])
plt.contourf(zzz)

In [ ]:
FEM2D = Scipy2DFEM(geom,FEM1Dx, FEM1Dy, folder=problem.folderName)

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec)
solver.set_save_slabs(200)
solver.set_Nt_max(1e6)
q_save, tt_save,  comp_time, err, err_vertex = solver.solve(stab_coeff=0.05,with_error = True, with_error_vertex = True)
print(err)


#### Tentative FEM (with DeC+SUPG)
* Weak formulation
$$
\begin{cases}
    \sum_K \int_K\left( \varphi_i +\alpha h_K \partial_x \varphi_i \right)\left( \partial_t u +\partial_x p\right)=0\\
    \sum_K \int_K\left( \varphi_i +\alpha h_K \partial_y \varphi_i \right)\left( \partial_t v +\partial_y p\right)=0\\
    \sum_K \int_K\left( \varphi_i +\alpha h_K \partial_x \varphi_i+\alpha h_K\partial_y \varphi_i \right)\left( \partial_t p +\partial_x u + \partial_y v \right)=0
\end{cases}
$$
* Let us define the residual quantities of each equation and its vector
$$
r(u,v,p):=
\begin{cases}
    r_u(u,v,p):= \partial_t u +\partial_x p\\
    r_v(u,v,p):= \partial_t v +\partial_y p\\
    r_v(u,v,p):= \partial_t p +\partial_x u +\partial_y v
\end{cases}
$$
and the time descrete equivalent in each subtime node $m=1,\dots, M$ and high order time discretization
$$
r^m(u,v,p):=\begin{cases}
    r^m_u(u,v,p):= \frac{u^m-u^0}{\Delta t} + \sum_r \theta^m_r \partial_x p^r\\
    r^m_v(u,v,p):= \frac{v^m-v^0}{\Delta t} +\sum_r \theta^m_r\partial_y p^r\\
    r^m_v(u,v,p):= \frac{p^m-p^0}{\Delta t} + \sum_r \theta^m_r (\partial_x u^r +\partial_y v^r)
\end{cases}
$$

* DeC on top of that
    * $\mathcal L^1$
    $$
        \mathcal L^1(u,v,p) = \begin{cases}
                |C_i| \frac{u_i^{m}-u_i^{0}}{\Delta t} + \sum_K \int_K \varphi_i \partial_x p^{0}\\
    |C_i| \frac{v_i^{m}-v_i^{0}}{\Delta t}+\sum_K \int_K \varphi_i  \partial_y p^{0}\\
    |C_i| \frac{p_i^{m}-p_i^{0}}{\Delta t} +\sum_K \int_K  \varphi_i   (\partial_x u^0 + \partial_y v^0) 
        \end{cases}
    $$
    * $\mathcal L^2$
    $$
        \mathcal L^{2,m}_i(u,v,p) = \int \left(\varphi_i + \alpha h \partial_x\varphi_i\, \text{sign}(J f^x) + \alpha h \partial_y \varphi_i\, \text{sign}(J f^y)   \right) r^m(u,v,p) dx 
    $$


In Einstein notation:
* Conservation Law $i = 1,\dots, N_{eq}$, $d=1,\dots,N_{\text{dim}}$
$$
\partial_t u^i + \partial_{x_d} f^{d,i}(u) =0.
$$
* SUPG option 1   $i,j,r, \ell = 1,\dots, N_{eq}$, $d,k=1,\dots,N_{\text{dim}}$
$$
\sum_K\int_K\left( \varphi^i + \alpha_{\ell j} \partial_{u^j} f^{k,r}(u) \partial_{x_k} \varphi^r \right) \left(\partial_t u^j + \partial_{x_d} f^{d,j}(u) \right)
$$

* SUPG option 2   $i,j,r = 1,\dots, N_{eq}$, $d,k=1,\dots,N_{\text{dim}}$
$$
\sum_K\int_K\left( \varphi^i + \alpha_{ij} \partial_{u^r} f^{k,j}(u) \partial_{x_k} \varphi^r \right) \left(\partial_t u^j + \partial_{x_d} f^{d,j}(u) \right)
$$

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec)
solver.set_save_slabs(200)
solver.set_Nt_max(1e6)
q_save, tt_save, comp_time, err, err_vertex = solver.solve(with_error = True)
print(err)
div_errors = FEM2D.compute_discrete_divergence_norms(q_save)

In [ ]:
solver = DeCSpaceTimeSUPGSolver(problem, FEM2D, dec, GF=True)
solver.set_save_slabs(200)
solver.set_Nt_max(1e6)
q_save_GF, tt_save_GF, comp_time_GF, err_GF, err_GF_vertex = solver.solve(with_error = True)
print(err_GF)
div_errors_GF = FEM2D.compute_discrete_divergence_norms(q_save_GF)


In [ ]:
plt.semilogy(tt_save, div_errors,label="SUPG")
plt.semilogy(tt_save_GF, div_errors_GF,label="SUPG-GF")
plt.legend()


In [ ]:
print(compute_integral_error(problem, FEM2D, q_save, tt_save))

Nt_save = len(tt_save)
err = np.zeros((Nt_save,len(problem.vars)))
rel_err = np.zeros((Nt_save,len(problem.vars)))
for it in range(Nt_save):
    err[it,:], rel_err[it,:] = compute_errors(problem, FEM2D, q_save, it, tt_save[it])

plt.figure()
plt.plot(tt_save[:-1], err[:-1], label=problem.vars)
plt.legend()
plt.title("Absolute errors")

# plt.figure()
# plt.plot(tt_save[:-1], rel_err[:-1], label=problem.vars)
# plt.legend()
# plt.title("Relative errors")
print(err)

#### Plot solutions

In [ ]:
levels = np.linspace(-1,1,14)

for it in range(0,200,20):
    plot_all_sols(problem, FEM2D, q_save, it, tt_save[it])#,levels)

In [ ]:
levels = np.linspace(-1,1,14)

for it in range(0,200,20):
    plot_all_sols(problem, FEM2D, q_save_GF, it, tt_save_GF[it])#,levels)

#### Plot exact

In [ ]:
for it in range(0,Nt_save):
    fig, axs = plt.subplots(1,problem.n_eq, figsize=(15,4))
    t = tt_save[it]
    for k, var in enumerate(problem.vars):
        ex = FEM2D.evaluate_function(lambda x,y: problem.exact[var](x,y,t))
        axs[k].set_title(var)
        fig.suptitle("Time %1.4f"%t)
        plot_sol(FEM2D,ex, axs[k],fig, levels)

#### Plot error

In [ ]:
for it in range(0,Nt_save,3):
    fig, axs = plt.subplots(1,problem.n_eq, figsize=(15,4))
    t = tt_save[it]
    for k, var in enumerate(problem.vars):
        ex = FEM2D.evaluate_function(lambda x,y: problem.exact[var](x,y,t))
        axs[k].set_title(var)
        plot_sol(FEM2D,q_save[var][it,:]-ex, axs[k],fig)
    fig.suptitle("Time %1.4f"%t)